In [5]:
import polars as pl
import logging
from pathlib import Path
from datetime import datetime
from google.colab import drive
drive.mount('/content/drive')

class SilverPipeline:
    def __init__(self, base_dir_path):
        self.base_dir = Path(base_dir_path)
        self.lakehouse_dir = self.base_dir / 'lakehouse'
        self.bronze_dir = self.lakehouse_dir / 'bronze'
        self.silver_dir = self.lakehouse_dir / 'silver'
        self.rejected_dir = self.lakehouse_dir / 'rejected_records'

        self.run_id = self._get_latest_bronze_run()
        self.latest_bronze = self.bronze_dir / self.run_id
        self.current_silver = self.silver_dir / self.run_id

        self.current_silver.mkdir(parents=True, exist_ok=True)
        self.rejected_dir.mkdir(parents=True, exist_ok=True)

        # Prevent the "Small Files" anti-pattern
        self.quarantine_buffer = []

    def _get_latest_bronze_run(self):
        runs = sorted([d for d in self.bronze_dir.iterdir() if d.is_dir()])
        if not runs:
            raise FileNotFoundError("No Bronze data found.")
        return runs[-1].name

    def load_bronze(self, filename, dtypes=None):
        file_path = self.latest_bronze / filename
        if not file_path.exists():
            logging.warning(f"File {filename} not found in {self.run_id}.")
            return pl.DataFrame()
        return pl.read_csv(file_path, schema_overrides=dtypes, infer_schema_length=10000)

    # ==========================================
    # DATA QUALITY
    # ==========================================
    def enforce_dq_rule(self, df: pl.DataFrame, dataset_name: str, rule_name: str, condition: pl.Expr) -> pl.DataFrame:
        if df.is_empty():
            return df

        valid_df = df.filter(condition)
        invalid_df = df.filter(~condition)

        if not invalid_df.is_empty():
            #  Buffer rejected records instead of immediately writing a tiny file
            invalid_df = invalid_df.with_columns(
                pl.lit(dataset_name).alias("Dataset_Name"),
                pl.lit(rule_name).alias("DQ_Failure_Reason"),
                pl.lit(datetime.now()).alias("Quarantine_Timestamp")
            )
            self.quarantine_buffer.append(invalid_df)
            logging.warning(f"[{dataset_name}] DQ Check '{rule_name}' Failed: {invalid_df.height} records quarantined.")

        return valid_df

    def enforce_deduplication(self, df: pl.DataFrame, dataset_name: str, unique_cols: list) -> pl.DataFrame:
        if df.is_empty():
            return df

        df_with_rn = df.with_row_index("temp_rn")
        valid_df_with_rn = df_with_rn.unique(subset=unique_cols, keep="first")
        invalid_df = df_with_rn.join(valid_df_with_rn, on="temp_rn", how="anti").drop("temp_rn")
        valid_df = valid_df_with_rn.drop("temp_rn")

        if not invalid_df.is_empty():
            invalid_df = invalid_df.with_columns(
                pl.lit(dataset_name).alias("Dataset_Name"),
                pl.lit("Duplicate_Record").alias("DQ_Failure_Reason"),
                pl.lit(datetime.now()).alias("Quarantine_Timestamp")
            )
            self.quarantine_buffer.append(invalid_df)
            logging.warning(f"[{dataset_name}] DQ Check 'Duplicates' Failed: {invalid_df.height} records quarantined.")

        return valid_df

    def flush_quarantine(self):
        """ Writes all buffered rejected records into a single file."""
        if not self.quarantine_buffer:
            logging.info("No records quarantined. Buffer is empty.")
            return

        try:
            # how="diagonal" handles datasets with different column schemas perfectly
            final_rejected_df = pl.concat(self.quarantine_buffer, how="diagonal")
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            rejected_file = self.rejected_dir / f"quarantine_log_{timestamp}.parquet"

            final_rejected_df.write_parquet(rejected_file)
            logging.info(f"Flushed {final_rejected_df.height} total quarantined records to {rejected_file.name}")
        except Exception as e:
            logging.error(f"Failed to flush quarantine buffer: {e}")

    # ==========================================
    # DATASET PROCESSORS
    # ==========================================
    def process_master_data(self, df: pl.DataFrame, dataset_name: str, id_col: str) -> list:
        if df.is_empty():
            return []

        logging.info(f"Applying DQ Checks to {dataset_name} Master...")
        df = self.enforce_deduplication(df, dataset_name, unique_cols=[id_col])
        df = self.enforce_dq_rule(df, dataset_name, f"Null_{id_col}", pl.col(id_col).is_not_null())
        df = self.enforce_dq_rule(df, dataset_name, f"Corrupt_Format_{id_col}", pl.col(id_col).cast(pl.Utf8).str.contains(r"^[A-Za-z0-9_-]+$"))

        df.write_parquet(self.current_silver / f"{dataset_name}_silver.parquet")
        logging.info(f"{dataset_name.capitalize()} Master Silver layer saved: {df.height} valid records.")

        return df[id_col].to_list()

    def process_transactions(self, txn_raw: pl.DataFrame, valid_outlets: list, valid_distributors: list):
        if txn_raw.is_empty():
            return

        logging.info("Applying DQ Checks to Transactions...")

        df = self.enforce_deduplication(txn_raw, "transactions", unique_cols=txn_raw.columns)
        df = self.enforce_dq_rule(df, "transactions", "Null_Volumes", pl.col("Volume_Liters").is_not_null())
        df = self.enforce_dq_rule(df, "transactions", "Ghost_Zero_Sales", pl.col("Volume_Liters") > 0)

        if valid_outlets:
            df = self.enforce_dq_rule(df, "transactions", "Orphan_Outlet", pl.col("Outlet_ID").is_in(valid_outlets))
        if valid_distributors:
            df = self.enforce_dq_rule(df, "transactions", "Orphan_Distributor", pl.col("Distributor_ID").is_in(valid_distributors))

        # Temporal Validation (Assuming standard 'Date' or 'Transaction_Date' column)
        date_col = "Date" if "Date" in df.columns else "Transaction_Date"
        if date_col in df.columns:
            try:
                # Ensure date is within 2020 and 2025 bounds
                df = df.with_columns(pl.col(date_col).cast(pl.Date, strict=False))
                df = self.enforce_dq_rule(df, "transactions", "Future_Or_Legacy_Date",
                                          pl.col(date_col).dt.year().is_between(2020, 2025))
            except Exception as e:
                logging.warning(f"Temporal validation skipped/failed: {e}")

        # Base Math / Business Logic Fix (Per-Outlet Windows)
        df = df.with_columns(
            pl.col("Volume_Liters").quantile(0.99).over("Outlet_ID").alias("Outlet_P99")
        )
        df = self.enforce_dq_rule(df, "transactions", "System_Artifact_Outlier", pl.col("Volume_Liters") <= pl.col("Outlet_P99"))
        df = df.drop("Outlet_P99")

        df.write_parquet(self.current_silver / "transactions_silver.parquet")
        logging.info(f"Transactions Silver layer saved: {df.height} valid records.")


# ==========================================
# EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s', force=True)
    print("🚀 Starting Silver Layer Lakehouse Processing...")

    try:
        BASE = '/content/drive/MyDrive/DataStorm_TeamZypher'
        pipeline = SilverPipeline(base_dir_path=BASE)

        print("\n⏳ Processing Master Data & Enforcing Type/Format Constraints...")
        outlet_raw = pipeline.load_bronze('outlet_master.csv', dtypes={'Outlet_ID': pl.Utf8})
        valid_outlets = pipeline.process_master_data(outlet_raw, "outlet_master", "Outlet_ID")

        #  Corrected Distributor Logic
        dist_details = pipeline.load_bronze('distributor_seasonality_details.csv', dtypes={'Distributor_ID': pl.Utf8})
        if not dist_details.is_empty() and "Distributor_ID" in dist_details.columns:
            valid_distributors = dist_details["Distributor_ID"].drop_nulls().unique().to_list()
            logging.info(f"Extracted {len(valid_distributors)} valid distributors from seasonality details.")
        else:
            valid_distributors = []

        print("\n⏳ Processing Transactions with Cross-Dataset Validation...")
        txn_raw = pipeline.load_bronze('transactions_history_final.csv', dtypes={
            'Outlet_ID': pl.Utf8,
            'Distributor_ID': pl.Utf8,
            'Volume_Liters': pl.Float64,
            'Total_Bill_Value': pl.Float64
        })

        pipeline.process_transactions(txn_raw, valid_outlets, valid_distributors)

        #Flush Quarantine Buffer
        print("\n⏳ Flushing Quarantine Buffer to Lakehouse...")
        pipeline.flush_quarantine()

        print("\n" + "="*60)
        print("✅ SUCCESS: SILVER LAYER PIPELINE COMPLETED!")
        print(f"📁 Clean data saved to: {pipeline.current_silver}")
        print(f"🗑️ Quarantined records saved to: {pipeline.rejected_dir}")
        print("="*60 + "\n")

    except Exception as e:
        print("\n❌ ERROR IN PIPELINE EXECUTION:")
        logging.error(str(e))

2026-05-16 23:09:33,428 - [INFO] - Applying DQ Checks to outlet_master Master...
2026-05-16 23:09:33,480 - [INFO] - Outlet_master Master Silver layer saved: 20000 valid records.
2026-05-16 23:09:33,492 - [INFO] - Extracted 10 valid distributors from seasonality details.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Starting Silver Layer Lakehouse Processing...

⏳ Processing Master Data & Enforcing Type/Format Constraints...

⏳ Processing Transactions with Cross-Dataset Validation...


2026-05-16 23:09:35,624 - [INFO] - Applying DQ Checks to Transactions...
2026-05-16 23:09:36,670 - [WARNING] - [transactions] DQ Check 'Ghost_Zero_Sales' Failed: 4853 records quarantined.
2026-05-16 23:09:37,187 - [WARNING] - [transactions] DQ Check 'System_Artifact_Outlier' Failed: 22656 records quarantined.
2026-05-16 23:09:37,692 - [INFO] - Transactions Silver layer saved: 2348880 valid records.
2026-05-16 23:09:37,857 - [INFO] - Flushed 27509 total quarantined records to quarantine_log_20260516_230937.parquet



⏳ Flushing Quarantine Buffer to Lakehouse...

✅ SUCCESS: SILVER LAYER PIPELINE COMPLETED!
📁 Clean data saved to: /content/drive/MyDrive/DataStorm_TeamZypher/lakehouse/silver/20260516_225859
🗑️ Quarantined records saved to: /content/drive/MyDrive/DataStorm_TeamZypher/lakehouse/rejected_records

